# Check and train the pancreas models

Run the data audit first. C1–C8 here check the environment, loader and models. When the project is ready for training, start with the pilot in C9. Each training and test action has its own switch.

For later GPU training, select a GPU runtime in Colab and check its actual availability in C5. [Colab runtime and Drive guidance](https://research.google.com/colaboratory/faq.html)

## C1. Project location

Put the project files in `My Drive/Major`, or change the Drive path in this cell. Use a Colab GPU runtime when you reach training.

In [ ]:
import os
import sys
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    raise RuntimeError("Open this notebook in Google Colab; local execution is disabled.")

# Change this Drive path if your project is in another Drive folder.
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Major" if IN_COLAB
    else os.environ.get("PANCREAS_PROJECT_ROOT", Path.cwd())
).expanduser()
if not IN_COLAB and PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT = PROJECT_ROOT.resolve()
print("Project folder:", PROJECT_ROOT)
print("Running in Colab:", IN_COLAB)

## C2. Connect Drive and find the project

Connect Drive to read the project files and save checkpoints and results.

In [ ]:
if IN_COLAB:
    drive.mount("/content/drive")

required_files = ("config.py", "check_environment.py", "prepare_data.py", "requirements.txt",
                  "dataset.py", "model.py", "predict.py", "train.py", "evaluate.py", "check_pipeline.py")
missing_files = [name for name in required_files if not (PROJECT_ROOT / name).is_file()]
if missing_files:
    raise FileNotFoundError(
        f"Missing {missing_files} in {PROJECT_ROOT}. "
        "Copy the project setup files there, or correct the path in C1."
    )

os.environ["PANCREAS_PROJECT_ROOT"] = str(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Project setup files found.")

## C3. Install the project libraries

Install into this notebook's Python environment. This command keeps already installed packages when their requirements are satisfied. If installation asks for a runtime restart, restart it and run C1–C5 again.

In [ ]:
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
    "-r", str(PROJECT_ROOT / "requirements.txt"),
])

## C4. Load the shared paths

The Python files and both notebooks use `config.py`. Only the folders needed for setup and data preparation are created now. Later model and training code will live in Python files imported by the notebooks.

In [ ]:
import importlib
import config

importlib.reload(config)
config.prepare_folders()
print("Source dataset:", config.DATASET_DIR)
print("Prepared data:", config.PROCESSED_DIR)
print("Split lists:", config.SPLITS_DIR)
print("Persistent results:", config.RESULTS_DIR)
print("Working dataset:", config.RUNTIME_DATASET_DIR)

## C5. Check imports and hardware

This runs the shared checker, including a small PyTorch operation. It saves actual versions, RAM, disk space, and CUDA status in `results/environment_local.json` or `results/environment_colab.json`. CPU setup can pass without a GPU; GPU execution is a separate check.

In [ ]:
import check_environment

importlib.reload(check_environment)
environment = check_environment.run_checks()
if not environment["checks_passed"]:
    raise RuntimeError(
        "Environment checks failed. Review the printed errors and saved JSON report."
    )

## C6. Check the expected dataset location

This checks file locations only. The real annotation, alignment, and patient-split audit belongs to Phase 3. Missing data does not prevent the setup checks above.

In [ ]:
expected_parts = (
    config.DATASET_DIR / "dataset.json",
    config.DATASET_DIR / "imagesTr",
    config.DATASET_DIR / "labelsTr",
    config.DATASET_DIR / "imagesTs",
)
for path in expected_parts:
    print(f"{path.name}: {'present' if path.exists() else 'missing'}")
dataset_present = all(path.exists() for path in expected_parts)
print("Expected dataset locations present:", dataset_present)
print("The data audit and split are in check_data.ipynb.")

## C7. Copy data to local runtime storage

Set `COPY_DATA_TO_RUNTIME` to `True` only when the dataset is ready. In Colab, this copies the extracted dataset from Drive to the VM after checking free space. It checks relative filenames and sizes before reusing a copy. It does not audit NIfTI contents or verify file hashes.

Use this copy if the runtime has enough disk space. If copying stops, rerun C7 to finish the missing files. Logs and checkpoints stay in Drive. C8 and training verify audit checksums for the volumes they use.

In [ ]:
import shutil

# Enable this after the dataset is present. The Drive source stays in place.
COPY_DATA_TO_RUNTIME = False
TRAINING_DATASET_DIR = config.DATASET_DIR

def file_sizes(folder):
    sizes = {}
    for path in folder.rglob("*"):
        if path.is_file():
            sizes[str(path.relative_to(folder))] = path.stat().st_size
    return sizes

if COPY_DATA_TO_RUNTIME and IN_COLAB:
    source = config.DATASET_DIR
    destination = config.RUNTIME_DATASET_DIR
    if not dataset_present:
        raise FileNotFoundError("Add dataset.json, imagesTr, labelsTr and imagesTs before copying.")

    source_files = file_sizes(source)
    if not source_files:
        raise ValueError("The source dataset folder is empty.")
    destination.parent.mkdir(parents=True, exist_ok=True)

    existing = file_sizes(destination)
    if existing.keys() - source_files.keys():
        raise ValueError("The VM copy has extra files. Restart the Colab session to make a fresh copy.")
    pending = [name for name, size in source_files.items() if existing.get(name) != size]
    partial = destination.parent / "dataset-copy.partial"
    partial.unlink(missing_ok=True)
    required_bytes = sum(source_files[name] for name in pending)
    if shutil.disk_usage(destination.parent).free < required_bytes * 1.1:
        raise OSError("Not enough runtime disk space to finish the dataset copy.")
    if pending:
        print(f"Copying {len(pending)} files to the Colab runtime.")
    for name in pending:
        target = destination / name
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(source / name, partial)
        partial.replace(target)

    if file_sizes(destination) != source_files:
        raise RuntimeError(
            "The runtime copy is incomplete or differs from the source. "
            "Use a fresh runtime copy before training."
        )
    TRAINING_DATASET_DIR = destination
    print("Runtime copy checked by relative filenames and sizes.")
elif COPY_DATA_TO_RUNTIME:
    print("Local session: use the dataset in the project folder.")
else:
    print("Runtime copy skipped. Enable it here when the dataset is available.")

print("Dataset path for later training:", TRAINING_DATASET_DIR)

## C8. Check the loader and the models

Finish the data notebook first. This step loads real training cases, checks a small tumour, and checks that the saved mask lines up with the original CT. It also checks the gradients and a weight reload for each model. The results go in `results/preflight/`.

In [ ]:
import json
import torch
from IPython.display import Image, display
from check_pipeline import run_preflight

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
preflight = run_preflight(TRAINING_DATASET_DIR, DEVICE)
print(json.dumps(preflight, indent=2))
for item in preflight["preparation"]:
    display(Image(filename=str(config.RESULTS_DIR / "preflight" / item["overlay"])))

## C9. Start the training pilot when the project is ready

Set `RUN_PILOT` to `True` when you are ready to train. Use a GPU runtime. This checks that U-Net learns on a small real batch, trains a short baseline, resumes it from Drive, then measures the full model on the same pilot subset. Look at the measured time and peak memory before choosing a budget.

In [ ]:
from train import run_pilot

RUN_PILOT = False
if RUN_PILOT:
    report = run_pilot(TRAINING_DATASET_DIR, DEVICE)
    print(json.dumps(report, indent=2))
else:
    print("Pilot is waiting for your start.")

## C10. Run the four comparisons

Review C8 and C9 before starting. Set `RUN_EPOCHS` once for the full comparison after timing the pilot. Choose a `RUN_NUMBER` from the printed list and run one model at a time. After a disconnect, use the same number and budget to resume.

In [ ]:
from train import experiment_runs, freeze_comparison, run_comparison, run_name

RUN_EPOCHS = config.TRAINING_EPOCHS  # Choose the shared budget after C9.
INCLUDE_REPEATS = False
EXPERIMENTS = experiment_runs(INCLUDE_REPEATS)

for number, (variant, seed) in enumerate(EXPERIMENTS, start=1):
    print(number, run_name(variant, seed))

RUN_NUMBER = None
if RUN_NUMBER is not None:
    if not 1 <= RUN_NUMBER <= len(EXPERIMENTS):
        raise ValueError("Use one of the run numbers printed above.")
    run_comparison(EXPERIMENTS, RUN_NUMBER - 1, TRAINING_DATASET_DIR, DEVICE, RUN_EPOCHS)

## C11. Freeze the checkpoints

Once the planned runs have all reached the chosen budget, set `FREEZE_RUNS` to `True`. The notebook records the exact checkpoint files and draws the training and validation curves. Model selection uses the validation scores.

In [ ]:
FREEZE_RUNS = False
if FREEZE_RUNS:
    manifest_path = freeze_comparison(EXPERIMENTS, epochs=RUN_EPOCHS)
    print(manifest_path)
    display(Image(filename=str(config.RESULTS_DIR / "training_curves.png")))

## C12. Evaluate the held-out cases

Set `RUN_TEST` to `True` after C11 succeeds. This runs inference on every labelled held-out CT and saves scores in `results/` and error overlays in `results/held_out/`. The original MSD `imagesTs` scans have no reference masks in this project.

In [ ]:
from evaluate import evaluate_test

RUN_TEST = False
if RUN_TEST:
    manifest_path = config.RESULTS_DIR / "frozen_comparison.json"
    if not manifest_path.is_file():
        raise FileNotFoundError("Freeze the completed comparisons in C11 first.")
    table = evaluate_test(TRAINING_DATASET_DIR, manifest_path, DEVICE)
    for row in table:
        print(row)